<a href="https://colab.research.google.com/github/VETTOORLIJO/DS_VettoorLijo/blob/master/moduleendassignmnet4_ml_OOPS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor)
from sklearn.svm import SVR
from sklearn.metrics import (mean_squared_error,mean_absolute_error,r2_score)


In [5]:
class CarPricePredictor:

    def __init__(self, file_path):

        self.file_path = file_path

        self.df = None
        self.X = None
        self.y = None

        self.X_train = None
        self.X_test = None

        self.y_train = None
        self.y_test = None

        self.X_train_scaled = None
        self.X_test_scaled = None

        self.scaler = StandardScaler()

        self.results = []

    # ==============================
    # Load Data
    # ==============================

    def load_data(self):

        self.df = pd.read_csv(self.file_path)

        print("Shape:", self.df.shape)
        print("Missing:", self.df.isna().sum().sum())
        print("Duplicates:", self.df.duplicated().sum())

    # ==============================
    # Clean Data
    # ==============================

    def clean_data(self):

        self.df["Car_brand"] = (
            self.df["CarName"]
            .str.split()
            .str[0]
            .str.lower()
        )

        brand_map = {
            "maxda": "mazda",
            "porcshce": "porsche",
            "toyouta": "toyota",
            "vokswagen": "volkswagen",
            "vw": "volkswagen"
        }

        self.df["Car_brand"] = (self.df["Car_brand"].replace(brand_map))

    # ==============================
    # Feature Engineering
    # ==============================

    def prepare_features(self):

        df_obj = self.df.select_dtypes(include="object").drop(["CarName", "enginelocation"], axis=1)
        df_obj_enc = pd.get_dummies( df_obj,drop_first=True,dtype=int)

        df_num = self.df.select_dtypes(include=np.number).drop(["car_ID"],axis=1 )

        df_final = pd.concat([df_num, df_obj_enc],axis=1)

        self.X = df_final.drop("price", axis=1)
        self.y = df_final["price"]

    # ==============================
    # Split Data
    # ==============================

    def split_data(self):

        (self.X_train,self.X_test,self.y_train,self.y_test) = train_test_split(self.X, self.y,test_size=0.2, random_state=42)

    # ==============================
    # Scaling
    # ==============================

    def scale_data(self):

        self.X_train_scaled = (self.scaler.fit_transform(self.X_train))

        self.X_test_scaled = (self.scaler.transform(self.X_test))

    # ==============================
    # Evaluation
    # ==============================

    def evaluate_model(self,y_pred,train_score,model_name):

        self.results.append({
            "Model": model_name,
            "MSE": mean_squared_error(self.y_test,y_pred),
            "MAE": mean_absolute_error(self.y_test,y_pred),
            "R2": r2_score(self.y_test,y_pred),
            "Train Score": train_score
        })

    # ==============================
    # Train Models
    # ==============================

    def train_linear_regression(self):

        model = LinearRegression()
        model.fit(self.X_train_scaled,self.y_train)

        y_pred = model.predict(self.X_test_scaled)
        self.evaluate_model(y_pred,model.score(self.X_train_scaled,self.y_train),"Linear Regression")

    def train_decision_tree(self):

        model = DecisionTreeRegressor(random_state=42)

        model.fit(self.X_train,self.y_train)

        y_pred = model.predict(self.X_test)

        self.evaluate_model(y_pred,model.score(self.X_train,self.y_train),"Decision Tree")

    def train_random_forest(self):

        self.rf = RandomForestRegressor(random_state=42)
        self.rf.fit(self.X_train,self.y_train)

        y_pred = self.rf.predict(self.X_test)

        self.evaluate_model(y_pred,self.rf.score(self.X_train,self.y_train),"Random Forest")

    def train_gradient_boosting(self):

        model = GradientBoostingRegressor(learning_rate=0.1,
                                          loss="absolute_error",
                                          max_depth=2,
                                          n_estimators=300,
                                          random_state=42)

        model.fit(self.X_train,self.y_train)

        y_pred = model.predict(self.X_test)

        self.evaluate_model(y_pred,model.score(self.X_train,self.y_train),"Gradient Boosting")

    def train_svr(self):

        model = SVR(C=100,epsilon=0.5,gamma="scale",kernel="linear")

        model.fit(self.X_train_scaled,self.y_train)

        y_pred = model.predict(self.X_test_scaled)

        self.evaluate_model(y_pred,model.score(self.X_train_scaled,self.y_train),"SVR")

    # ==============================
    # Results
    # ==============================

    def show_results(self):

        results_df = pd.DataFrame(self.results)

        results_df = results_df.sort_values("R2",ascending=False)

        print(results_df)

    # ==============================
    # Feature Importance
    # ==============================

    def feature_importance(self):

        importance = pd.DataFrame({"Feature":self.X_train.columns,"Importance":self.rf.feature_importances_ })

        importance = (importance.sort_values("Importance",ascending=False))

        print(importance.head(10))

    # ==============================
    # Save Model
    # ==============================

    def save_model(self):

        bundle = {"model": self.rf,"feature_cols":self.X.columns.tolist()}

        joblib.dump(bundle,"car_price_rf_bundle.joblib")

        print("Model Saved")

In [8]:
car = CarPricePredictor("CarPrice_Assignment.csv")

In [10]:
car.load_data()

Shape: (205, 26)
Missing: 0
Duplicates: 0


In [12]:
car.df.head()

,car_ID,symboling,CarName,fueltype,aspiration,doornumber,carbody,drivewheel,enginelocation,wheelbase,...,enginesize,fuelsystem,boreratio,stroke,compressionratio,horsepower,peakrpm,citympg,highwaympg,price
0,1,3,alfa-romero giulia,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,13495.0
1,2,3,alfa-romero stelvio,gas,std,two,convertible,rwd,front,88.6,...,130,mpfi,3.47,2.68,9.0,111,5000,21,27,16500.0
2,3,1,alfa-romero Quadrifoglio,gas,std,two,hatchback,rwd,front,94.5,...,152,mpfi,2.68,3.47,9.0,154,5000,19,26,16500.0
3,4,2,audi 100 ls,gas,std,four,sedan,fwd,front,99.8,...,109,mpfi,3.19,3.40,10.0,102,5500,24,30,13950.0
4,5,2,audi 100ls,gas,std,four,sedan,4wd,front,99.4,...,136,mpfi,3.19,3.40,8.0,115,5500,18,22,17450.0


In [13]:
car.df.columns

Index(['car_ID', 'symboling', 'CarName', 'fueltype', 'aspiration',
       'doornumber', 'carbody', 'drivewheel', 'enginelocation', 'wheelbase',
       'carlength', 'carwidth', 'carheight', 'curbweight', 'enginetype',
       'cylindernumber', 'enginesize', 'fuelsystem', 'boreratio', 'stroke',
       'compressionratio', 'horsepower', 'peakrpm', 'citympg', 'highwaympg',
       'price'],
      dtype='object')

In [14]:
car.df.dtypes.value_counts()

,count
object,10
int64,8
float64,8


In [15]:
car.clean_data()

In [16]:
car.prepare_features()
car.split_data()
car.scale_data()

car.train_linear_regression()
car.train_decision_tree()
car.train_random_forest()
car.train_gradient_boosting()
car.train_svr()

car.show_results()

car.feature_importance()

car.save_model()

               Model           MSE          MAE        R2  Train Score
2      Random Forest  3.314851e+06  1287.478650  0.958010     0.985154
3  Gradient Boosting  5.960761e+06  1712.034561  0.924494     0.965763
0  Linear Regression  7.128547e+06  1763.565986  0.909701     0.974704
1      Decision Tree  8.775123e+06  2024.227634  0.888844     0.998654
4                SVR  1.183166e+07  2202.374018  0.850126     0.944323
          Feature  Importance
6      enginesize    0.555923
5      curbweight    0.294317
13     highwaympg    0.046281
10     horsepower    0.026732
3        carwidth    0.013976
43  Car_brand_bmw    0.007967
2       carlength    0.006948
1       wheelbase    0.006931
12        citympg    0.005625
11        peakrpm    0.005066
Model Saved
